## 1)Load Model, PCA, Scaler

In [1]:
import pickle, torch, numpy as np
from sentence_transformers import SentenceTransformer

# Load preprocessing
pca = pickle.load(open("pca_transform.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))

# Model
class MLP(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(dim, 256), 
            torch.nn.ReLU(), 
            torch.nn.Dropout(0.2),
            torch.nn.Linear(256, 64), 
            torch.nn.ReLU(), 
            torch.nn.Dropout(0.1),
            torch.nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model = MLP(68)
model.load_state_dict(torch.load("suspicion_model.pt", map_location="cpu"))
model.eval()
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Model Loaded.")

Model Loaded.


## 2)Prediction Function

In [2]:
def predict(record, threshold=0.65):

    text = (
        str(record.get('title', '')) + " " +
        str(record.get('description', '')) + " " +
        str(record.get('category', ''))
    )
    
    emb = embedder.encode([text])
    emb_p = pca.transform(emb)

    price = float(record.get("price", 0))
    msrp = float(record.get("msrp", 0))
    paz = float(record.get("price_anomaly_z", 0))
    sim = float(record.get("clip_image_text_sim", 0))
    sim_norm = (sim + 1) / 2

    num = np.array([[price, msrp, paz, sim_norm]])
    num_s = scaler.transform(num)

    X = np.hstack([emb_p, num_s]).astype(np.float32)

    with torch.no_grad():
        p = torch.sigmoid(model(torch.tensor(X))).item()

    p = max(min(p, 1), 0)  # clip NaN/inf

    return {
        "suspicion_score": round(p, 4),
        "label": "suspicious" if p >= threshold else "safe"
    }

## 3)Example Predictions

In [4]:
test_samples = [
    {
        "title": "Apple AirPods Pro",
        "description": "Active noise cancellation, original Apple product with warranty.",
        "category": "Electronics",
        "price": 19990,
        "msrp": 24900,
        "price_anomaly_z": 0.12,
        "clip_image_text_sim": 0.91
    },
    {
        "title": "Gucci Women's Luxury Leather Handbag",
        "description": "Imported leather bag for women. No brand authentication provided.",
        "category": "Fashion",
        "price": 899,
        "msrp": 82000,
        "price_anomaly_z": -4.8,
        "clip_image_text_sim": 0.12
    },
    {
        "title": "Samsung 65-inch Crystal 4K UHD Smart TV",
        "description": "2023 model, genuine Samsung product, 1-year warranty.",
        "category": "Electronics",
        "price": 62999,
        "msrp": 78990,
        "price_anomaly_z": -0.5,
        "clip_image_text_sim": 0.88
    },
    {
        "title": "Apple iPhone 15 Pro Max",
        "description": "Brand new iPhone. Cheapest price online.",
        "category": "Electronics",
        "price": 1299,
        "msrp": 159900,
        "price_anomaly_z": -9.5,
        "clip_image_text_sim": 0.21
    },
    {
        "title": "L’Oréal Paris Revitalift Anti-Ageing Cream",
        "description": "Official L'Oréal skin cream for women.",
        "category": "Beauty",
        "price": 799,
        "msrp": 999,
        "price_anomaly_z": -0.15,
        "clip_image_text_sim": 0.72
    },
    {
        "title": "Dior Sauvage Eau de Parfum for Men",
        "description": "Premium perfume at the best price. Imported.",
        "category": "Beauty",
        "price": 599,
        "msrp": 9500,
        "price_anomaly_z": -4.1,
        "clip_image_text_sim": 0.19
    },
    {
        "title": "Nike Air Zoom Pegasus 40 Running Shoes",
        "description": "Original Nike running shoes for athletes.",
        "category": "Footwear",
        "price": 8895,
        "msrp": 10499,
        "price_anomaly_z": -0.41,
        "clip_image_text_sim": 0.83
    },
    {
        "title": "Adidas Ultraboost 22 Shoes",
        "description": "Top-quality imported shoes for men.",
        "category": "Footwear",
        "price": 699,
        "msrp": 18999,
        "price_anomaly_z": -3.9,
        "clip_image_text_sim": 0.27
    },
    {
        "title": "Rolex Submariner Stainless Steel Watch",
        "description": "Luxury watch for men. Premium build.",
        "category": "Accessories",
        "price": 1499,
        "msrp": 785000,
        "price_anomaly_z": -8.1,
        "clip_image_text_sim": 0.16
    },
    {
        "title": "Philips OneBlade Hybrid Trimmer",
        "description": "Original Philips product with fast charging.",
        "category": "Electronics",
        "price": 1999,
        "msrp": 2399,
        "price_anomaly_z": -0.22,
        "clip_image_text_sim": 0.87
    }
]

# Run predictions
for i, sample in enumerate(test_samples, start=1):
    print(f"\nSample {i}: {sample['title']}")
    print(predict(sample))


Sample 1: Apple AirPods Pro
{'suspicion_score': 0.0001, 'label': 'safe'}

Sample 2: Gucci Women's Luxury Leather Handbag
{'suspicion_score': 1.0, 'label': 'suspicious'}

Sample 3: Samsung 65-inch Crystal 4K UHD Smart TV
{'suspicion_score': 1.0, 'label': 'suspicious'}

Sample 4: Apple iPhone 15 Pro Max
{'suspicion_score': 1.0, 'label': 'suspicious'}

Sample 5: L’Oréal Paris Revitalift Anti-Ageing Cream
{'suspicion_score': 0.0023, 'label': 'safe'}

Sample 6: Dior Sauvage Eau de Parfum for Men
{'suspicion_score': 1.0, 'label': 'suspicious'}

Sample 7: Nike Air Zoom Pegasus 40 Running Shoes
{'suspicion_score': 0.0034, 'label': 'safe'}

Sample 8: Adidas Ultraboost 22 Shoes
{'suspicion_score': 1.0, 'label': 'suspicious'}

Sample 9: Rolex Submariner Stainless Steel Watch
{'suspicion_score': 1.0, 'label': 'suspicious'}

Sample 10: Philips OneBlade Hybrid Trimmer
{'suspicion_score': 0.0016, 'label': 'safe'}


C:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler wa